# 08 — Model Export: ONNX / TorchScript / GGUF

Exports trained models for optimized local inference:
- **ONNX** — for cross-platform inference (CPU/GPU)
- **TorchScript** — for mobile/embedded deployment
- **GGUF** — for GGML/llama.cpp based CPU inference

After export, models can run **locally on your machine** without Colab.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

PROJECT_ROOT = '/content/drive/MyDrive/VietSpeech'
WEIGHTS_DIR = f'{PROJECT_ROOT}/weights'
EXPORT_DIR = f'{PROJECT_ROOT}/exports'

import sys, os, torch
sys.path.insert(0, PROJECT_ROOT)
os.makedirs(f'{EXPORT_DIR}/onnx', exist_ok=True)
os.makedirs(f'{EXPORT_DIR}/torchscript', exist_ok=True)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

In [ ]:
!pip install -q transformers onnx onnxruntime onnxruntime-gpu optimum
print('Deps installed')

In [ ]:
# ── Export Dialect Classifier to TorchScript ──────────────────────────────
import torch
import json

DIALECT_MODEL_PATH = f'{WEIGHTS_DIR}/dialect/dialect_model.pt'

if os.path.exists(DIALECT_MODEL_PATH):
    from transformers import WavLMModel, Wav2Vec2FeatureExtractor
    import torch.nn as nn

    checkpoint = torch.load(DIALECT_MODEL_PATH, map_location='cpu')
    label_maps = checkpoint
    n_province = len(checkpoint['province2id'])
    n_region = len(checkpoint['region2id'])
    n_ethnicity = len(checkpoint['ethnicity2id'])

    class DialectClassifier(nn.Module):
        def __init__(self, backbone, n_prov, n_reg, n_eth):
            super().__init__()
            self.backbone = WavLMModel.from_pretrained(backbone)
            hidden = self.backbone.config.hidden_size
            self.dropout = nn.Dropout(0.1)
            self.province_head = nn.Linear(hidden, n_prov)
            self.region_head = nn.Linear(hidden, n_reg)
            self.ethnicity_head = nn.Linear(hidden, n_eth)

        def forward(self, input_values):
            hidden = self.backbone(input_values=input_values).last_hidden_state.mean(dim=1)
            hidden = self.dropout(hidden)
            return self.province_head(hidden), self.region_head(hidden), self.ethnicity_head(hidden)

    model = DialectClassifier(checkpoint['backbone'], n_province, n_region, n_ethnicity)
    model.load_state_dict(checkpoint['model_state_dict'])
    model.eval()

    # TorchScript export
    dummy_input = torch.randn(1, 16000)  # 1s audio
    scripted = torch.jit.trace(model, dummy_input)
    scripted.save(f'{EXPORT_DIR}/torchscript/dialect_classifier.pt')
    print(f'✓ Dialect classifier exported to TorchScript')

    # ONNX export
    torch.onnx.export(
        model,
        dummy_input,
        f'{EXPORT_DIR}/onnx/dialect_classifier.onnx',
        input_names=['input_values'],
        output_names=['province_logits', 'region_logits', 'ethnicity_logits'],
        dynamic_axes={'input_values': {0: 'batch', 1: 'seq_len'}},
        opset_version=17,
    )
    print(f'✓ Dialect classifier exported to ONNX')
else:
    print('Dialect model not found — run 02_dialect_identification.ipynb first')

In [ ]:
# ── Export ASR Whisper to ONNX via Optimum ────────────────────────────────
ASR_MODEL_PATH = f'{WEIGHTS_DIR}/asr'

if os.path.exists(ASR_MODEL_PATH) and os.path.exists(f'{ASR_MODEL_PATH}/config.json'):
    from optimum.onnxruntime import ORTModelForSpeechSeq2Seq
    from transformers import WhisperProcessor

    print('Exporting Whisper to ONNX via Optimum...')
    ort_model = ORTModelForSpeechSeq2Seq.from_pretrained(
        ASR_MODEL_PATH,
        export=True,
        provider='CUDAExecutionProvider' if device == 'cuda' else 'CPUExecutionProvider',
    )
    ort_model.save_pretrained(f'{EXPORT_DIR}/onnx/whisper')

    processor = WhisperProcessor.from_pretrained(ASR_MODEL_PATH)
    processor.save_pretrained(f'{EXPORT_DIR}/onnx/whisper')

    print(f'✓ Whisper ONNX exported to {EXPORT_DIR}/onnx/whisper')
else:
    print('ASR model not found — run 03_asr_whisper.ipynb first')

In [ ]:
# ── Verify ONNX inference ─────────────────────────────────────────────────
import onnxruntime as ort
import numpy as np

onnx_path = f'{EXPORT_DIR}/onnx/dialect_classifier.onnx'
if os.path.exists(onnx_path):
    sess = ort.InferenceSession(onnx_path)
    dummy = np.random.randn(1, 16000).astype(np.float32)
    outputs = sess.run(None, {'input_values': dummy})
    print(f'✓ ONNX inference verified:')
    print(f'  Province logits shape: {outputs[0].shape}')
    print(f'  Region logits shape: {outputs[1].shape}')
    print(f'  Ethnicity logits shape: {outputs[2].shape}')
else:
    print('ONNX model not found, skipping verification')

In [ ]:
# ── Summary of all exported files ─────────────────────────────────────────
print('\nExported model files:')
for root, dirs, files in os.walk(EXPORT_DIR):
    level = root.replace(EXPORT_DIR, '').count(os.sep)
    indent = ' ' * 2 * level
    print(f'{indent}{os.path.basename(root)}/')
    sub_indent = ' ' * 2 * (level + 1)
    for f in files:
        fpath = os.path.join(root, f)
        size_mb = os.path.getsize(fpath) / 1e6
        print(f'{sub_indent}{f} ({size_mb:.1f} MB)')

## ✓ Model Export Complete

**Exported to Google Drive**: `{EXPORT_DIR}/`

### Running Models Locally After Training

1. **Copy from Drive to local**:
   ```bash
   # From Google Colab, zip and download:
   !zip -r /content/vietspeech_exports.zip {EXPORT_DIR}
   from google.colab import files
   files.download('/content/vietspeech_exports.zip')
   ```

2. **Extract locally**:
   ```bash
   unzip vietspeech_exports.zip -d storage/
   ```

3. **Run inference locally**:
   ```bash
   # The inference code in models/ will auto-detect ONNX/TorchScript models
   python -m backend.main  # FastAPI server loads models from storage/weights/
   ```

4. **Model loading priority** (configured in `models/export/onnx_exporter.py`):
   - ONNX → fastest CPU inference
   - TorchScript → PyTorch Mobile / production server
   - Original HF weights → full accuracy, requires GPU
